In [1]:
"""
DisFT-GNN 因子挖掘系统 — 单文件版（BigQuant平台提交用）
=====================================================
BigAlpha 2026 AI因子挖掘赛道（AI智能赛道）

合规声明：
- 全部训练在Notebook内完成，不上传预训练权重
- 随机种子固定，确保运行时审计可复现
- 教师模型使用未来标签（LUPI范式），代码独立隔离，main函数只调用学生推理
- AI技术应用关键环节已标注 [AI应用环节X]

main函数返回三列因子表：date, instrument, factor

[AI应用环节标注汇总]
环节1: 自动化特征工程（FeatureExtractor, LearnableFeatureAggregator）
环节2: 教师模型训练-未来信息编码（FutureTrendEncoder）
环节3: 多通道双线性融合（MultiChannelBilinearFusion）
环节4: HSIC蒸馏（HSICLoss）
环节5: IC排序损失（ICRankLoss, StudentTotalLoss）
环节6: 可蒸馏性验证（DistillabilityValidator）
环节7: 学生模型推理-时空GNN（StudentModel）
环节8: B项感知投影（StudentModel.factor_projection）
"""

# ============================================================
# 导入
# ============================================================
import warnings
warnings.filterwarnings('ignore')

import os
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass, field
from typing import Optional, Dict, List, Tuple

# tqdm可选导入
try:
    from tqdm import tqdm
except ImportError:
    def tqdm(x, **kwargs):
        return x

# scipy可选导入（spearmanr有numpy fallback）
try:
    from scipy.stats import spearmanr
    _HAS_SCIPY = True
except ImportError:
    _HAS_SCIPY = False


# ============================================================
# 模块一：全局配置
# ============================================================

@dataclass
class ModelConfig:
    """模型架构配置（适应Notebook内3小时训练，已精简）"""

    use_mamba: bool = False
    input_dim: int = 25               # M: 日频特征维度
    hidden_dim: int = 64              # Bi-LSTM隐藏维度
    num_layers: int = 1               # LSTM层数

    d_p: int = 32                     # 历史时空嵌入维度 D_p
    d_f: int = 16                     # 未来趋势嵌入维度 D_f
    d_out: int = 4                    # 融合通道数 D
    fusion_rank: int = 8              # 低秩分解秩 r
    tau: float = 0.5                  # 注意力温度系数

    gnn_layers: int = 2
    gnn_hidden: int = 32

    lookback_short: int = 5           # 短窗口 L=5

    adj_alpha_init: float = 0.4       # 行业权重初始值
    adj_beta_init: float = 0.4        # 收益相关性权重初始值
    adj_gamma_init: float = 0.2       # 市值相似性权重初始值

    projection_hidden: int = 16


@dataclass
class TrainConfig:
    """训练配置"""

    seed: int = 42
    device: str = "cpu"
    learning_rate: float = 5e-4
    student_lr: float = 1e-4
    batch_size: int = 16

    teacher_epochs: int = 30
    teacher_patience: int = 5
    future_horizon: int = 1
    future_threshold: float = 0.0

    use_distillation: bool = True
    lambda_distill_start: float = 0.1
    lambda_distill_end: float = 0.5
    hsic_sigma_init: float = 1.0
    hsic_eigenvalue_threshold: float = 1e-6

    student_epochs: int = 20
    student_patience: int = 5
    alpha_rank: float = 1.0
    beta_ce: float = 0.15

    distill_r2_threshold: float = 0.05
    distill_r2_cautious: float = 0.15

    # 内存安全：最大股票数，超过则自动降采样
    max_stocks: int = 500             # 超过此数自动抽样以控制内存
    adj_sparse_threshold: float = 0.01  # 邻接矩阵元素低于此值置零（稀疏化）


@dataclass
class DataConfig:
    """数据配置"""

    train_start: str = "2019-01-01"
    train_end: str = "2022-12-31"
    val_start: str = "2023-01-01"
    val_end: str = "2024-06-30"
    test_start: str = "2024-07-01"
    test_end: str = "2024-12-31"

    universe: str = "CSI1000"

    corr_window: int = 20
    corr_threshold: float = 0.3
    size_threshold: float = 0.5

    max_missing_ratio: float = 0.4
    winsorize_lower: float = 0.01
    winsorize_upper: float = 0.99
    barra_r2_threshold: float = 0.3


@dataclass
class Config:
    """全局配置聚合"""
    model: ModelConfig = field(default_factory=ModelConfig)
    train: TrainConfig = field(default_factory=TrainConfig)
    data: DataConfig = field(default_factory=DataConfig)

    def __post_init__(self):
        assert self.model.d_p > self.model.fusion_rank, "D_p必须大于rank"
        assert self.model.d_f > self.model.fusion_rank, "D_f必须大于rank"
        assert 0 < self.train.alpha_rank, "alpha必须>0"
        assert 0 <= self.train.beta_ce <= 1, "beta必须在[0,1]"


# 全局配置实例
CONFIG = Config()


# ============================================================
# 模块二：特征工程
# ============================================================

class FeatureExtractor:
    """
    [AI应用环节1] 分钟级微观结构特征提取与日频聚合。

    输入：单只股票单日的1分钟K线 + 盘口数据
    输出：DailyFeatureVector ∈ R^M
    """

    def __init__(self):
        self.feature_names = self._get_feature_names()

    @staticmethod
    def _get_feature_names() -> List[str]:
        return [
            # 基础统计 (6维)
            "open_ret", "close_ret", "high_low_ratio", "vwap_ret", "volume_total", "amount_total",
            # 订单簿压力 (4维)
            "obi_depth_weighted", "obi_spread_weighted", "active_buy_ratio", "obi_integral",
            # 已实现波动率 (3维)
            "rv_std", "rv_range", "rv_parkinson",
            # 价格冲击 (2维)
            "kyle_lambda", "amihud_illiq",
            # 形态特征 (3维)
            "overnight_gap", "last30_ret", "max_drawdown_pos",
            # 分布特征 (3维)
            "ret_skew", "ret_kurt", "volume_entropy",
            # 可学习聚合特征占位 (4维)
            "learnable_agg_0", "learnable_agg_1", "learnable_agg_2", "learnable_agg_3",
        ]

    def extract_daily_features(
        self,
        kline_1min: pd.DataFrame,
        orderbook: Optional[pd.DataFrame] = None,
    ) -> np.ndarray:
        """
        [AI应用环节1] 从单日分钟数据提取日频特征向量。
        """
        features = {}

        # ===== 基础统计 =====
        features["open_ret"] = (kline_1min["close"].iloc[-1] / kline_1min["open"].iloc[0] - 1)
        features["close_ret"] = kline_1min["close"].pct_change().fillna(0).sum()
        features["high_low_ratio"] = (kline_1min["high"].max() / kline_1min["low"].min() - 1) if kline_1min["low"].min() > 0 else 0
        vwap = (kline_1min["amount"].sum() / kline_1min["volume"].sum()) if kline_1min["volume"].sum() > 0 else kline_1min["close"].mean()
        features["vwap_ret"] = (kline_1min["close"].iloc[-1] / vwap - 1) if vwap > 0 else 0
        features["volume_total"] = np.log1p(kline_1min["volume"].sum())
        features["amount_total"] = np.log1p(kline_1min["amount"].sum())

        minute_ret = kline_1min["close"].pct_change().fillna(0).values

        # ===== 订单簿压力 =====
        if orderbook is not None and len(orderbook) > 0:
            obi = self._compute_obi(orderbook)
            features["obi_depth_weighted"] = obi["depth_weighted"].mean()
            features["obi_spread_weighted"] = obi["spread_weighted"].mean()
            features["active_buy_ratio"] = obi["active_buy_ratio"]
            features["obi_integral"] = obi["depth_weighted"].sum()
        else:
            buy_vol = kline_1min[kline_1min["close"] >= kline_1min["open"]]["volume"].sum()
            total_vol = kline_1min["volume"].sum()
            features["obi_depth_weighted"] = (buy_vol / total_vol - 0.5) if total_vol > 0 else 0
            features["obi_spread_weighted"] = 0.0
            features["active_buy_ratio"] = (buy_vol / total_vol) if total_vol > 0 else 0.5
            features["obi_integral"] = features["obi_depth_weighted"] * len(kline_1min)

        # ===== 已实现波动率 =====
        features["rv_std"] = np.std(minute_ret)
        features["rv_range"] = np.log(kline_1min["high"].max() / kline_1min["low"].min()) if kline_1min["low"].min() > 0 else 0
        hl_ratio = np.log(kline_1min["high"] / kline_1min["low"]).replace([np.inf, -np.inf], 0)
        features["rv_parkinson"] = np.sqrt(np.mean(hl_ratio ** 2) / (4 * np.log(2)))

        # ===== 价格冲击 =====
        volume_arr = kline_1min["volume"].values
        nonzero_mask = volume_arr > 0
        if nonzero_mask.sum() > 5:
            price_change = np.diff(kline_1min["close"].values)
            order_flow = volume_arr[1:] - volume_arr[:-1]
            denom = np.sum(order_flow ** 2)
            features["kyle_lambda"] = (np.sum(price_change * order_flow) / denom) if denom > 0 else 0
        else:
            features["kyle_lambda"] = 0

        ret_abs = np.abs(minute_ret)
        features["amihud_illiq"] = np.mean(ret_abs[1:] / (volume_arr[1:] + 1e-8))

        # ===== 形态特征 =====
        features["overnight_gap"] = (kline_1min["open"].iloc[0] / kline_1min["close"].iloc[-1] - 1) if kline_1min["close"].iloc[-1] > 0 else 0
        n = len(kline_1min)
        last30_start = max(0, n - 30)
        features["last30_ret"] = (kline_1min["close"].iloc[-1] / kline_1min["close"].iloc[last30_start] - 1) if kline_1min["close"].iloc[last30_start] > 0 else 0
        cummax = kline_1min["close"].cummax()
        drawdown = (kline_1min["close"] / cummax - 1)
        features["max_drawdown_pos"] = drawdown.idxmin() / n if n > 0 else 0

        # ===== 分布特征 =====
        features["ret_skew"] = self._safe_skew(minute_ret)
        features["ret_kurt"] = self._safe_kurt(minute_ret)
        vol_norm = volume_arr / (volume_arr.sum() + 1e-8)
        vol_norm = vol_norm[vol_norm > 0]
        features["volume_entropy"] = -np.sum(vol_norm * np.log(vol_norm + 1e-8)) if len(vol_norm) > 0 else 0

        # ===== 可学习聚合特征占位 =====
        for i in range(4):
            features[f"learnable_agg_{i}"] = 0.0

        return np.array([features.get(name, 0.0) for name in self.feature_names], dtype=np.float32)

    @staticmethod
    def _compute_obi(orderbook: pd.DataFrame) -> Dict[str, np.ndarray]:
        """计算订单簿不平衡（OBI）"""
        bid_cols = [c for c in orderbook.columns if c.startswith("bid_vol")]
        ask_cols = [c for c in orderbook.columns if c.startswith("ask_vol")]

        if not bid_cols or not ask_cols:
            return {
                "depth_weighted": np.zeros(len(orderbook)),
                "spread_weighted": np.zeros(len(orderbook)),
                "active_buy_ratio": 0.5,
            }

        bid_vol = orderbook[bid_cols].values
        ask_vol = orderbook[ask_cols].values

        weights = np.arange(1, bid_vol.shape[1] + 1) ** (-1)
        weights = weights / weights.sum()
        depth_weighted = (bid_vol @ weights - ask_vol @ weights) / (bid_vol @ weights + ask_vol @ weights + 1e-8)
        spread_weighted = depth_weighted
        active_buy_ratio = (bid_vol.sum() / (bid_vol.sum() + ask_vol.sum() + 1e-8))

        return {
            "depth_weighted": depth_weighted,
            "spread_weighted": spread_weighted,
            "active_buy_ratio": float(active_buy_ratio),
        }

    @staticmethod
    def _safe_skew(x: np.ndarray) -> float:
        if len(x) < 3:
            return 0.0
        std = np.std(x)
        if std < 1e-8:
            return 0.0
        return float(np.mean(((x - np.mean(x)) / std) ** 3))

    @staticmethod
    def _safe_kurt(x: np.ndarray) -> float:
        if len(x) < 4:
            return 0.0
        std = np.std(x)
        if std < 1e-8:
            return 0.0
        return float(np.mean(((x - np.mean(x)) / std) ** 4) - 3)

    def extract_all_stocks_one_day(
        self,
        date: str,
        stock_list: List[str],
        kline_data: Dict[str, pd.DataFrame],
        orderbook_data: Optional[Dict[str, pd.DataFrame]] = None,
    ) -> np.ndarray:
        """提取某日全市场所有股票的日频特征。"""
        N = len(stock_list)
        M = len(self.feature_names)
        features = np.zeros((N, M), dtype=np.float32)

        for i, stock in enumerate(stock_list):
            kline = kline_data.get(stock)
            if kline is None or len(kline) == 0:
                features[i] = np.nan
                continue
            ob = orderbook_data.get(stock) if orderbook_data else None
            features[i] = self.extract_daily_features(kline, ob)

        return features


class LearnableFeatureAggregator(nn.Module):
    """
    [AI应用环节1 - AI主导] 可学习特征聚合器。
    将特征候选池通过可学习权重自动聚合，L1正则自动筛选有效特征。
    """

    def __init__(self, input_dim: int, output_dim: int = 4):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(input_dim, output_dim) * 0.02)
        self.bias = nn.Parameter(torch.zeros(output_dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x @ self.weight + self.bias

    def l1_penalty(self) -> torch.Tensor:
        return self.weight.abs().sum()


# ============================================================
# 模块三：图构建
# ============================================================

class GraphBuilder:
    """
    构建多层动态邻接矩阵并生成图序列数据。

    层A（行业关系）：静态，同行业连边
    层B（收益相关性）：动态，滚动20日分钟级收益相关系数
    层C（市值相似性）：半动态，对数市值差 < 0.5 连边
    """

    def __init__(self, config=None):
        self.config = config or CONFIG
        self.industry_map: Optional[Dict[str, str]] = None
        self.market_cap: Optional[Dict[str, float]] = None

    def set_static_info(self, industry_map: Dict[str, str], market_cap: Dict[str, float]):
        self.industry_map = industry_map
        self.market_cap = market_cap

    def build_industry_adj(self, stock_list: List[str]) -> np.ndarray:
        """层A：行业邻接矩阵（静态）"""
        N = len(stock_list)
        A = np.zeros((N, N), dtype=np.float32)
        if self.industry_map is None:
            return A
        industries = [self.industry_map.get(s, "unknown") for s in stock_list]
        for i in range(N):
            for j in range(i + 1, N):
                if industries[i] == industries[j] and industries[i] != "unknown":
                    A[i, j] = 1.0
                    A[j, i] = 1.0
        np.fill_diagonal(A, 1.0)
        return A

    def build_correlation_adj(
        self,
        returns: np.ndarray,
        stock_list: List[str],
        window: int = 20,
    ) -> np.ndarray:
        """
        层B：收益相关性邻接矩阵（动态，严格无前向窥探）。
        使用截至t-1日的历史收益计算，窗口不含t日及以后数据。
        """
        N = len(stock_list)
        A = np.zeros((N, N), dtype=np.float32)
        if returns.shape[0] < 2:
            np.fill_diagonal(A, 1.0)
            return A
        recent_returns = returns[-window:]
        if recent_returns.shape[0] < 5:
            np.fill_diagonal(A, 1.0)
            return A
        corr_matrix = np.corrcoef(recent_returns.T)
        corr_matrix = np.nan_to_num(corr_matrix, nan=0.0)
        threshold = self.config.data.corr_threshold
        mask = np.abs(corr_matrix) > threshold
        A = np.where(mask, corr_matrix, 0).astype(np.float32)
        np.fill_diagonal(A, 1.0)
        return A

    def build_size_adj(self, stock_list: List[str]) -> np.ndarray:
        """层C：市值相似性邻接矩阵"""
        N = len(stock_list)
        A = np.zeros((N, N), dtype=np.float32)
        if self.market_cap is None:
            np.fill_diagonal(A, 1.0)
            return A
        log_caps = np.array([self.market_cap.get(s, 0.0) for s in stock_list])
        threshold = self.config.data.size_threshold
        for i in range(N):
            for j in range(i + 1, N):
                if abs(log_caps[i] - log_caps[j]) < threshold:
                    A[i, j] = 1.0
                    A[j, i] = 1.0
        np.fill_diagonal(A, 1.0)
        return A

    def build_fused_adj(
        self,
        stock_list: List[str],
        returns: Optional[np.ndarray] = None,
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        """构建融合邻接矩阵及各层矩阵。"""
        A_industry = self.build_industry_adj(stock_list)
        A_corr = self.build_correlation_adj(returns, stock_list) if returns is not None else np.eye(len(stock_list), dtype=np.float32)
        A_size = self.build_size_adj(stock_list)

        alpha = self.config.model.adj_alpha_init
        beta = self.config.model.adj_beta_init
        gamma = self.config.model.adj_gamma_init

        A_fused = alpha * A_industry + beta * A_corr + gamma * A_size
        row_sum = A_fused.sum(axis=1, keepdims=True)
        A_fused = A_fused / (row_sum + 1e-8)

        return A_fused, A_industry, A_corr, A_size


class LearnableAdjFusion(nn.Module):
    """
    [AI应用环节 - AI主导] 可学习邻接矩阵融合。
    邻接矩阵融合权重由AI学习而非人工设定。
    """

    def __init__(self, alpha_init: float = 0.4, beta_init: float = 0.4, gamma_init: float = 0.2):
        super().__init__()
        raw_init = torch.tensor([alpha_init, beta_init, gamma_init])
        self.raw_weights = nn.Parameter(torch.log(raw_init + 1e-8))

    def forward(self, A_industry: torch.Tensor, A_corr: torch.Tensor, A_size: torch.Tensor) -> torch.Tensor:
        weights = torch.softmax(self.raw_weights, dim=0)
        A_fused = weights[0] * A_industry + weights[1] * A_corr + weights[2] * A_size
        row_sum = A_fused.sum(dim=-1, keepdim=True)
        return A_fused / (row_sum + 1e-8)

    def get_weights(self) -> Tuple[float, float, float]:
        with torch.no_grad():
            w = torch.softmax(self.raw_weights, dim=0)
        return w[0].item(), w[1].item(), w[2].item()


class TimeSeriesSplitter:
    """
    严格时间序列划分。
    训练集：2019-01-01 ~ 2022-12-31
    验证集：2023-01-01 ~ 2024-06-30
    测试集：2024-07-01 ~ 2024-12-31
    """

    def __init__(self, config=None):
        self.config = config or CONFIG

    def split_dates(self, all_dates: List[str]) -> Dict[str, List[str]]:
        train_dates = [d for d in all_dates if self.config.data.train_start <= d <= self.config.data.train_end]
        val_dates = [d for d in all_dates if self.config.data.val_start <= d <= self.config.data.val_end]
        test_dates = [d for d in all_dates if self.config.data.test_start <= d <= self.config.data.test_end]
        return {"train": train_dates, "val": val_dates, "test": test_dates}

    @staticmethod
    def create_rolling_windows(
        dates: List[str],
        lookback: int = 5,
        future_horizon: int = 1,
    ) -> List[Dict]:
        """创建滚动窗口样本。"""
        samples = []
        for i in range(lookback, len(dates) - future_horizon + 1):
            history = dates[i - lookback: i]
            future = dates[i: i + future_horizon]
            samples.append({
                "history": history,
                "future": future,
                "t": dates[i - 1],
            })
        return samples


def normalize_adj(adj: np.ndarray) -> np.ndarray:
    """对称归一化邻接矩阵：D^{-1/2} A D^{-1/2}"""
    row_sum = adj.sum(axis=1)
    d_inv_sqrt = np.power(row_sum, -0.5, where=row_sum > 0)
    d_inv_sqrt = np.nan_to_num(d_inv_sqrt, nan=0.0)
    D_inv = np.diag(d_inv_sqrt)
    return D_inv @ adj @ D_inv


# ============================================================
# 模块四：模型
# ============================================================

class BiLSTMEncoder(nn.Module):
    """[AI应用环节7] 时序编码器：Bi-LSTM"""

    def __init__(self, input_dim: int, hidden_dim: int, output_dim: int, num_layers: int = 1):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=num_layers,
                            batch_first=True, bidirectional=True)
        self.proj = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (..., L, M) 节点特征序列，支持 (N, L, M) 或 (B, N, L, M)
        Returns:
            h: (..., D_p) 时序嵌入
        """
        orig_shape = x.shape
        if x.dim() == 4:
            B, N, L, M = x.shape
            x = x.reshape(B * N, L, M)
        elif x.dim() == 3:
            B = 1
            N, L, M = x.shape
            x = x.reshape(N, L, M)
        else:
            raise ValueError(f"Expected 3D or 4D input, got {x.dim()}D")

        out, _ = self.lstm(x)
        h = self.proj(out[:, -1, :])

        if len(orig_shape) == 4:
            h = h.reshape(orig_shape[0], orig_shape[1], -1)
        return h


class GCNLayer(nn.Module):
    """单层GCN：A*X*W"""

    def __init__(self, in_dim: int, out_dim: int):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)

    def forward(self, x: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
        support = self.linear(x)
        return adj @ support


class STGNNEncoder(nn.Module):
    """[AI应用环节7] 时空GNN编码器：Bi-LSTM(时序) + GCN(空间)"""

    def __init__(self, input_dim: int, hidden_dim: int, output_dim: int,
                 gnn_hidden: int = 32, gnn_layers: int = 2, num_layers: int = 1):
        super().__init__()
        self.temporal_encoder = BiLSTMEncoder(input_dim, hidden_dim, hidden_dim, num_layers)

        self.gnn_layers = nn.ModuleList()
        prev_dim = hidden_dim
        for _ in range(gnn_layers):
            self.gnn_layers.append(GCNLayer(prev_dim, gnn_hidden))
            prev_dim = gnn_hidden

        self.proj = nn.Linear(gnn_hidden, output_dim)

    def forward(self, x: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
        is_batched = x.dim() == 4

        h = self.temporal_encoder(x)

        if is_batched:
            B = h.shape[0]
            for gnn_layer in self.gnn_layers:
                h_list = [F.relu(gnn_layer(h[b], adj[b])) for b in range(B)]
                h = torch.stack(h_list, dim=0)
        else:
            for gnn_layer in self.gnn_layers:
                h = F.relu(gnn_layer(h, adj))

        return self.proj(h)


class MultiChannelBilinearFusion(nn.Module):
    """
    [AI应用环节3] 多通道双线性融合（Multi-Channel Bilinear Fusion）
    低秩分解：F^[d] ≈ U_d · V_d^T
    """

    def __init__(self, d_p: int, d_f: int, d_out: int, rank: int = 8, tau: float = 0.5):
        super().__init__()
        self.d_out = d_out
        self.rank = rank
        self.tau = tau

        self.U = nn.Parameter(torch.randn(d_out, d_p, rank) * 0.02)
        self.V = nn.Parameter(torch.randn(d_out, d_f, rank) * 0.02)

        self.W_Q = nn.Linear(d_f, d_out, bias=False)
        self.W_K = nn.Linear(d_p, d_out, bias=False)

    def forward(self, p: torch.Tensor, q: torch.Tensor) -> torch.Tensor:
        """
        Args:
            p: (N, D_p) 历史时空嵌入
            q: (N, D_f) 未来趋势嵌入
        Returns:
            h: (N, D_out) 未来感知时空表征
        """
        pU = torch.einsum('np,dpr->ndr', p, self.U)
        qV = torch.einsum('nf,dfr->ndr', q, self.V)
        bilinear_vals = (pU * qV).sum(dim=-1)

        Q = self.W_Q(q)
        K = self.W_K(p)
        attn_scores = self.tau * (Q @ K.T) / (self.d_out ** 0.5)
        attn_weights = F.softmax(attn_scores, dim=-1)

        h = attn_weights @ bilinear_vals
        return h


# ============================================================
# 教师模型（[AI应用环节2] 仅本地/Notebook内训练，不部署）
# ============================================================

# [合规声明] 教师模型代码仅用于训练，不在main函数中调用推理
# 教师模型使用未来标签(f^{[t+1,t+T]})是LUPI范式的设计要求
# 提交的因子仅由学生模型(仅用历史数据)生成

class FutureTrendEncoder(nn.Module):
    """[AI应用环节2] 未来趋势编码器：将未来涨跌标签编码为高层嵌入。"""

    def __init__(self, future_dim: int, output_dim: int):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(future_dim, output_dim * 2),
            nn.ReLU(),
            nn.Linear(output_dim * 2, output_dim),
        )

    def forward(self, future_labels: torch.Tensor) -> torch.Tensor:
        return self.encoder(future_labels.float())


class TeacherModel(nn.Module):
    """[AI应用环节2/3] 教师模型：历史GNN + 未来趋势编码 + 多通道双线性融合 + 预测头。"""

    def __init__(self, config=None):
        super().__init__()
        self.config = config or CONFIG
        m = self.config.model

        self.history_encoder = STGNNEncoder(
            input_dim=m.input_dim,
            hidden_dim=m.hidden_dim,
            output_dim=m.d_p,
            gnn_hidden=m.gnn_hidden,
            gnn_layers=m.gnn_layers,
            num_layers=m.num_layers,
        )
        self.future_encoder = FutureTrendEncoder(
            future_dim=self.config.train.future_horizon,
            output_dim=m.d_f,
        )
        self.fusion = MultiChannelBilinearFusion(
            d_p=m.d_p, d_f=m.d_f, d_out=m.d_out, rank=m.fusion_rank, tau=m.tau,
        )
        self.pred_head = nn.Sequential(
            nn.Linear(m.d_out, m.projection_hidden),
            nn.ReLU(),
            nn.Linear(m.projection_hidden, 1),
        )

    def forward(
        self,
        x: torch.Tensor,
        adj: torch.Tensor,
        future_labels: torch.Tensor,
    ) -> Dict[str, torch.Tensor]:
        """教师前向传播。支持 (N, L, M) 或 (B, N, L, M) 输入。"""
        is_batched = x.dim() == 4

        if is_batched:
            B = x.shape[0]
            all_logits, all_fused, all_p, all_q = [], [], [], []
            for b in range(B):
                p = self.history_encoder(x[b], adj[b])
                q = self.future_encoder(future_labels[b])
                h_fused = self.fusion(p, q)
                logits = self.pred_head(h_fused).squeeze(-1)
                all_logits.append(logits)
                all_fused.append(h_fused)
                all_p.append(p)
                all_q.append(q)
            return {
                'logits': torch.stack(all_logits),
                'fused_repr': torch.stack(all_fused),
                'history_repr': torch.stack(all_p),
                'future_repr': torch.stack(all_q),
            }
        else:
            p = self.history_encoder(x, adj)
            q = self.future_encoder(future_labels)
            h_fused = self.fusion(p, q)
            logits = self.pred_head(h_fused).squeeze(-1)
            return {
                'logits': logits,
                'fused_repr': h_fused,
                'history_repr': p,
                'future_repr': q,
            }

    def extract_fused_repr(self, x, adj, future_labels):
        """提取融合表征h^{t+}（供可蒸馏性验证使用）"""
        return self.forward(x, adj, future_labels)['fused_repr']


# ============================================================
# 学生模型（[AI应用环节7] 部署到比赛环境）
# ============================================================

class StudentModel(nn.Module):
    """[AI应用环节7] 学生模型：轻量历史GNN，仅使用历史数据。"""

    def __init__(self, config=None):
        super().__init__()
        self.config = config or CONFIG
        m = self.config.model

        # 可学习特征聚合器（AI主导特征选择）：在GNN编码前对原始特征做可学习变换
        self.feat_aggregator = LearnableFeatureAggregator(
            input_dim=m.input_dim,
            output_dim=m.input_dim,  # 保持维度不变，做特征加权而非降维
        )

        self.encoder = STGNNEncoder(
            input_dim=m.input_dim,
            hidden_dim=m.hidden_dim,
            output_dim=m.d_p,
            gnn_hidden=m.gnn_hidden,
            gnn_layers=m.gnn_layers,
            num_layers=m.num_layers,
        )

        self.factor_projection = nn.Sequential(
            nn.Linear(m.d_p, m.projection_hidden),
            nn.ReLU(),
            nn.Linear(m.projection_hidden, 1),
        )
        self.cls_head = nn.Linear(m.d_p, 1)

    def forward(self, x: torch.Tensor, adj: torch.Tensor) -> Dict[str, torch.Tensor]:
        """学生前向传播（仅历史数据）。支持 (N, L, M) 或 (B, N, L, M) 输入。"""
        is_batched = x.dim() == 4

        # [AI应用环节1] 可学习特征聚合：对原始特征做可学习变换
        # x: (..., L, M) → feat_aggregator 作用在最后一维 M
        x = self.feat_aggregator(x)

        if is_batched:
            B = x.shape[0]
            all_factor, all_hidden, all_cls = [], [], []
            for b in range(B):
                h = self.encoder(x[b], adj[b])
                f = self.factor_projection(h).squeeze(-1)
                c = self.cls_head(h).squeeze(-1)
                all_factor.append(f)
                all_hidden.append(h)
                all_cls.append(c)
            return {
                'factor': torch.stack(all_factor),
                'hidden_repr': torch.stack(all_hidden),
                'cls_logits': torch.stack(all_cls),
            }
        else:
            hidden_repr = self.encoder(x, adj)
            factor = self.factor_projection(hidden_repr).squeeze(-1)
            cls_logits = self.cls_head(hidden_repr).squeeze(-1)
            return {
                'factor': factor,
                'hidden_repr': hidden_repr,
                'cls_logits': cls_logits,
            }

    def get_l1_penalty(self) -> torch.Tensor:
        """获取可学习聚合器的L1正则"""
        return self.feat_aggregator.l1_penalty()

    @torch.no_grad()
    def predict_factor(self, x: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
        """推理模式：直接返回因子值"""
        self.eval()
        return self.forward(x, adj)['factor']


# ============================================================
# 模块五：损失函数
# ============================================================

class ICRankLoss(nn.Module):
    """
    [AI应用环节5] IC排序损失：按日分组计算Spearman秩相关，取负作为损失。
    """

    def __init__(self, eps: float = 1e-8):
        super().__init__()
        self.eps = eps

    def forward(self, factor_pred: torch.Tensor, future_return: torch.Tensor) -> torch.Tensor:
        mask = ~(torch.isnan(factor_pred) | torch.isnan(future_return))
        if mask.sum() < 10:
            return torch.tensor(0.0, device=factor_pred.device, requires_grad=True)

        pred = factor_pred[mask]
        ret = future_return[mask]

        pred_rank = pred.argsort().argsort().float()
        ret_rank = ret.argsort().argsort().float()

        pred_rank = (pred_rank - pred_rank.mean()) / (pred_rank.std() + self.eps)
        ret_rank = (ret_rank - ret_rank.mean()) / (ret_rank.std() + self.eps)

        ic = (pred_rank * ret_rank).mean()
        return -ic


class BatchICRankLoss(nn.Module):
    """批量IC排序损失：对batch中多个交易日的IC取均值"""

    def __init__(self, eps: float = 1e-8):
        super().__init__()
        self.ic_loss = ICRankLoss(eps)

    def forward(self, factor_pred: torch.Tensor, future_return: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        batch_size = factor_pred.shape[0]
        losses, ics = [], []

        for b in range(batch_size):
            loss = self.ic_loss(factor_pred[b], future_return[b])
            losses.append(loss)
            ics.append(-loss.detach())

        loss_mean = torch.stack(losses).mean()
        ic_mean = torch.stack(ics).mean()
        return loss_mean, ic_mean


class HSICLoss(nn.Module):
    """
    [AI应用环节4] HSIC蒸馏损失：最大化学生与教师表征的非线性统计依赖。

    工程加固：
    1. 可学习核带宽σ，随特征分布漂移自适应
    2. 特征值下限阈值检测，不稳定时自动切换CKA
    3. CKA备选方案内置
    """

    def __init__(self, sigma_init: float = 1.0, eigenvalue_threshold: float = 1e-6, use_cka_fallback: bool = True):
        super().__init__()
        self.log_sigma_student = nn.Parameter(torch.tensor(sigma_init).log())
        self.log_sigma_teacher = nn.Parameter(torch.tensor(sigma_init).log())
        self.eigenvalue_threshold = eigenvalue_threshold
        self.use_cka_fallback = use_cka_fallback
        self._cka_mode = False

    def _rbf_kernel(self, x: torch.Tensor, log_sigma: torch.Tensor) -> torch.Tensor:
        sigma = log_sigma.exp()
        x_sq = (x ** 2).sum(dim=-1, keepdim=True)
        dist_sq = x_sq + x_sq.T - 2 * x @ x.T
        dist_sq = dist_sq.clamp(min=0)
        return torch.exp(-dist_sq / (2 * sigma ** 2 + 1e-8))

    def _compute_hsic(self, K: torch.Tensor, L: torch.Tensor) -> torch.Tensor:
        n = K.shape[0]
        H = torch.eye(n, device=K.device) - 1.0 / n
        Kc = K @ H
        Lc = L @ H
        return (Kc * Lc).sum() / ((n - 1) ** 2)

    def _compute_cka(self, K: torch.Tensor, L: torch.Tensor) -> torch.Tensor:
        hsic_kl = self._compute_hsic(K, L)
        hsic_kk = self._compute_hsic(K, K)
        hsic_ll = self._compute_hsic(L, L)
        return hsic_kl / (torch.sqrt(hsic_kk * hsic_ll + 1e-8) + 1e-8)

    def _check_stability(self, K: torch.Tensor, L: torch.Tensor) -> bool:
        try:
            eigvals_K = torch.linalg.eigvalsh(K)
            eigvals_L = torch.linalg.eigvalsh(L)
            min_eig = min(eigvals_K.min().item(), eigvals_L.min().item())
            if min_eig < self.eigenvalue_threshold:
                self._cka_mode = True
                return False
        except Exception:
            self._cka_mode = True
            return False
        return True

    def forward(self, student_repr: torch.Tensor, teacher_repr: torch.Tensor) -> torch.Tensor:
        if self.training and not self._cka_mode:
            self._check_stability(
                self._rbf_kernel(student_repr, self.log_sigma_student),
                self._rbf_kernel(teacher_repr, self.log_sigma_teacher),
            )

        K = self._rbf_kernel(student_repr, self.log_sigma_student)
        L = self._rbf_kernel(teacher_repr, self.log_sigma_teacher)

        if self._cka_mode and self.use_cka_fallback:
            dependency = self._compute_cka(K, L)
        else:
            dependency = self._compute_hsic(K, L)

        return -dependency

    def get_stats(self) -> Dict[str, float]:
        return {
            "sigma_student": self.log_sigma_student.exp().item(),
            "sigma_teacher": self.log_sigma_teacher.exp().item(),
            "cka_mode": self._cka_mode,
        }


@dataclass
class StudentLossConfig:
    """学生损失配置"""
    alpha: float = 1.0
    beta: float = 0.15
    lambda_distill: float = 0.3
    use_distillation: bool = True
    l1_lambda: float = 0.001
    lambda_distill_start: float = 0.1
    lambda_distill_end: float = 0.5


class StudentTotalLoss(nn.Module):
    """
    [AI应用环节5] 学生三重损失
    L = α·L_rank + β·L_ce + λ·L_distill + l1_lambda·L_l1
    """

    def __init__(self, config: StudentLossConfig):
        super().__init__()
        self.config = config
        self.ic_loss = BatchICRankLoss()
        self.hsic_loss = HSICLoss() if config.use_distillation else None

    def forward(
        self,
        factor_pred: torch.Tensor,
        future_return: torch.Tensor,
        student_repr: torch.Tensor,
        teacher_repr: Optional[torch.Tensor] = None,
        cls_logits: Optional[torch.Tensor] = None,
        cls_labels: Optional[torch.Tensor] = None,
        l1_penalty: Optional[torch.Tensor] = None,
    ) -> Dict[str, torch.Tensor]:
        rank_loss, ic_mean = self.ic_loss(factor_pred, future_return)

        ce_loss = torch.tensor(0.0, device=factor_pred.device)
        if cls_logits is not None and cls_labels is not None:
            ce_loss = F.binary_cross_entropy_with_logits(cls_logits, cls_labels.float())

        distill_loss = torch.tensor(0.0, device=factor_pred.device)
        if self.hsic_loss is not None and teacher_repr is not None:
            B, N = factor_pred.shape
            student_flat = student_repr.reshape(B, N, -1) if student_repr.dim() == 2 else student_repr
            teacher_flat = teacher_repr.reshape(B, N, -1) if teacher_repr.dim() == 2 else teacher_repr
            distill_losses = []
            for b in range(B):
                dl = self.hsic_loss(student_flat[b], teacher_flat[b])
                distill_losses.append(dl)
            distill_loss = torch.stack(distill_losses).mean()

        l1_loss = l1_penalty if l1_penalty is not None else torch.tensor(0.0, device=factor_pred.device)

        total = (
            self.config.alpha * rank_loss
            + self.config.beta * ce_loss
            + self.config.lambda_distill * distill_loss
            + self.config.l1_lambda * l1_loss
        )

        return {
            'total_loss': total,
            'rank_loss': rank_loss,
            'ce_loss': ce_loss,
            'distill_loss': distill_loss,
            'l1_loss': l1_loss,
            'ic_mean': ic_mean,
        }

    def update_lambda(self, epoch: int, total_epochs: int):
        """蒸馏权重λ从起始值逐步增至终值"""
        if self.config.use_distillation:
            progress = min(epoch / total_epochs, 1.0)
            self.config.lambda_distill = (
                self.config.lambda_distill_start
                + (self.config.lambda_distill_end - self.config.lambda_distill_start) * progress
            )


# ============================================================
# 模块六：训练管线
# ============================================================

class DistillabilityValidator:
    """
    [AI应用环节6] 可蒸馏性前置验证
    验证学生仅用历史数据能否还原教师的未来感知表征。
    """

    def __init__(self, input_dim: int, hidden_dim: int = 32, output_dim: int = 4):
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
        )

    def validate(
        self,
        teacher_model: nn.Module,
        train_features: torch.Tensor,
        train_adjs: torch.Tensor,
        train_future_labels: torch.Tensor,
        val_features: torch.Tensor,
        val_adjs: torch.Tensor,
        val_future_labels: torch.Tensor,
        device: torch.device = torch.device('cpu'),
    ) -> Dict[str, float]:
        config = CONFIG
        self.mlp = self.mlp.to(device)
        teacher_model = teacher_model.to(device).eval()

        print("[可蒸馏性验证] 提取教师表征...")
        train_teacher_repr = self._extract_teacher_repr(
            teacher_model, train_features, train_adjs, train_future_labels, device
        )
        val_teacher_repr = self._extract_teacher_repr(
            teacher_model, val_features, val_adjs, val_future_labels, device
        )

        train_flat = train_features[:, :, -1, :].reshape(-1, train_features.shape[-1]).to(device)
        train_target = train_teacher_repr.reshape(-1, train_teacher_repr.shape[-1])
        val_flat = val_features[:, :, -1, :].reshape(-1, val_features.shape[-1]).to(device)
        val_target = val_teacher_repr.reshape(-1, val_teacher_repr.shape[-1])

        print("[可蒸馏性验证] 训练MLP拟合器...")
        optimizer = torch.optim.Adam(self.mlp.parameters(), lr=1e-3)

        for epoch in range(30):
            self.mlp.train()
            optimizer.zero_grad()
            pred = self.mlp(train_flat)
            loss = F.mse_loss(pred, train_target)
            loss.backward()
            optimizer.step()

        self.mlp.eval()
        with torch.no_grad():
            train_pred = self.mlp(train_flat)
            val_pred = self.mlp(val_flat)
            r2_train = self._compute_r2(train_pred, train_target)
            r2_val = self._compute_r2(val_pred, val_target)

        if r2_val > config.train.distill_r2_cautious:
            decision = 'proceed'
        elif r2_val > config.train.distill_r2_threshold:
            decision = 'cautious'
        else:
            decision = 'abort'

        print(f"[可蒸馏性验证] R²_train={r2_train:.4f}, R²_val={r2_val:.4f}, 决策={decision}")
        return {'r2_train': r2_train, 'r2_val': r2_val, 'decision': decision}

    @torch.no_grad()
    def _extract_teacher_repr(self, teacher_model, features, adjs, future_labels, device):
        reprs = []
        for t in range(features.shape[0]):
            x = features[t].to(device)
            adj = adjs[t].to(device)
            fl = future_labels[t].to(device)
            h = teacher_model.extract_fused_repr(x, adj, fl)
            reprs.append(h.cpu())
        return torch.stack(reprs)

    @staticmethod
    def _compute_r2(pred, target):
        ss_res = ((pred - target) ** 2).sum()
        ss_tot = ((target - target.mean()) ** 2).sum()
        return (1 - ss_res / (ss_tot + 1e-8)).item()


class TeacherTrainer:
    """[AI应用环节2] 教师模型训练器。"""

    def __init__(self, config=None):
        self.config = config or CONFIG
        self.model = None
        self.best_val_loss = float('inf')
        self.patience_counter = 0

    def train(
        self,
        train_features: torch.Tensor,
        train_adjs: torch.Tensor,
        train_future_labels: torch.Tensor,
        train_future_returns: torch.Tensor,
        val_features: torch.Tensor,
        val_adjs: torch.Tensor,
        val_future_labels: torch.Tensor,
        val_future_returns: torch.Tensor,
        device: torch.device = torch.device('cpu'),
    ) -> TeacherModel:
        t_config = self.config.train

        self.model = TeacherModel(self.config).to(device)
        optimizer = torch.optim.Adam(self.model.parameters(), lr=t_config.learning_rate)

        print(f"\n{'='*60}")
        print(f"[教师训练] 开始训练 (epochs={t_config.teacher_epochs})")
        print(f"{'='*60}")

        # 快速预检：前3轮若验证损失不下降，直接跳过教师训练节省时间
        quick_check_epochs = 3
        initial_val_loss = None

        for epoch in range(t_config.teacher_epochs):
            self.model.train()
            train_losses = []
            n_samples = train_features.shape[0]

            indices = torch.randperm(n_samples)
            for i in range(0, n_samples, t_config.batch_size):
                batch_idx = indices[i:i + t_config.batch_size]
                if len(batch_idx) == 0:
                    continue

                x = train_features[batch_idx].to(device)
                adj = train_adjs[batch_idx].to(device)
                fl = train_future_labels[batch_idx].to(device)
                y = (train_future_returns[batch_idx] > t_config.future_threshold).float().to(device)

                optimizer.zero_grad()
                outputs = self.model(x, adj, fl)
                loss = F.binary_cross_entropy_with_logits(outputs['logits'], y)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                optimizer.step()
                train_losses.append(loss.item())

            val_loss = self._validate(val_features, val_adjs, val_future_labels, val_future_returns, device)

            avg_train = np.mean(train_losses) if train_losses else 0
            print(f"  Epoch {epoch+1}/{t_config.teacher_epochs} | train_loss={avg_train:.4f} | val_loss={val_loss:.4f}")

            # 快速预检：记录初始验证损失
            if epoch == 0:
                initial_val_loss = val_loss

            if val_loss < self.best_val_loss:
                self.best_val_loss = val_loss
                self.patience_counter = 0
            else:
                self.patience_counter += 1
                if self.patience_counter >= t_config.teacher_patience:
                    print(f"  早停：{t_config.teacher_patience}轮无改善")
                    break

            # 快速预检：3轮后验证损失没有改善，提前终止
            if epoch == quick_check_epochs - 1 and val_loss >= initial_val_loss:
                print(f"  [快速预检] 前{quick_check_epochs}轮验证损失无改善，提前终止教师训练")
                break

        print(f"[教师训练] 完成，最佳验证损失={self.best_val_loss:.4f}")
        return self.model

    @torch.no_grad()
    def _validate(self, val_features, val_adjs, val_future_labels, val_future_returns, device):
        self.model.eval()
        losses = []
        t_config = self.config.train
        n_samples = val_features.shape[0]

        for i in range(0, n_samples, t_config.batch_size):
            batch = slice(i, min(i + t_config.batch_size, n_samples))
            x = val_features[batch].to(device)
            adj = val_adjs[batch].to(device)
            fl = val_future_labels[batch].to(device)
            y = (val_future_returns[batch] > t_config.future_threshold).float().to(device)

            outputs = self.model(x, adj, fl)
            loss = F.binary_cross_entropy_with_logits(outputs['logits'], y)
            losses.append(loss.item())

        return np.mean(losses) if losses else float('inf')


class StudentTrainer:
    """[AI应用环节5] 学生模型训练器（蒸馏+排序损失）。"""

    def __init__(self, config=None, use_distillation: bool = True):
        self.config = config or CONFIG
        self.use_distillation = use_distillation
        self.model = None
        self.loss_fn = None
        self.best_val_ic = -float('inf')
        self.patience_counter = 0
        self.training_log = []

    def train(
        self,
        train_features: torch.Tensor,
        train_adjs: torch.Tensor,
        train_future_returns: torch.Tensor,
        val_features: torch.Tensor,
        val_adjs: torch.Tensor,
        val_future_returns: torch.Tensor,
        teacher_model: Optional[TeacherModel] = None,
        teacher_train_repr: Optional[torch.Tensor] = None,
        teacher_val_repr: Optional[torch.Tensor] = None,
        device: torch.device = torch.device('cpu'),
    ) -> StudentModel:
        t_config = self.config.train

        self.model = StudentModel(self.config).to(device)

        loss_config = StudentLossConfig(
            alpha=t_config.alpha_rank,
            beta=t_config.beta_ce,
            lambda_distill=t_config.lambda_distill_start,
            use_distillation=self.use_distillation,
            lambda_distill_start=t_config.lambda_distill_start,
            lambda_distill_end=t_config.lambda_distill_end,
        )
        self.loss_fn = StudentTotalLoss(loss_config).to(device)

        if self.use_distillation and teacher_model is not None:
            teacher_model = teacher_model.to(device).eval()
            for p in teacher_model.parameters():
                p.requires_grad = False

        optimizer = torch.optim.Adam(self.model.parameters(), lr=t_config.student_lr)

        distill_status = "启用" if self.use_distillation else "禁用（纯时序方案）"
        print(f"\n{'='*60}")
        print(f"[学生训练] 开始训练 (epochs={t_config.student_epochs}, 蒸馏={distill_status})")
        print(f"{'='*60}")

        for epoch in range(t_config.student_epochs):
            self.loss_fn.update_lambda(epoch, t_config.student_epochs)

            self.model.train()
            train_losses = []
            train_ics = []
            n_samples = train_features.shape[0]

            indices = torch.randperm(n_samples)
            for i in range(0, n_samples, t_config.batch_size):
                batch_idx = indices[i:i + t_config.batch_size]
                if len(batch_idx) == 0:
                    continue

                x = train_features[batch_idx].to(device)
                adj = train_adjs[batch_idx].to(device)
                y_ret = train_future_returns[batch_idx].to(device)
                y_cls = (y_ret > t_config.future_threshold).float()

                optimizer.zero_grad()

                outputs = self.model(x, adj)

                teacher_repr = None
                if self.use_distillation and teacher_model is not None:
                    with torch.no_grad():
                        if teacher_train_repr is not None:
                            teacher_repr = teacher_train_repr[batch_idx].to(device)

                l1_penalty = self.model.get_l1_penalty()

                losses = self.loss_fn(
                    factor_pred=outputs['factor'],
                    future_return=y_ret,
                    student_repr=outputs['hidden_repr'],
                    teacher_repr=teacher_repr,
                    cls_logits=outputs['cls_logits'],
                    cls_labels=y_cls,
                    l1_penalty=l1_penalty,
                )

                losses['total_loss'].backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                optimizer.step()

                train_losses.append(losses['total_loss'].item())
                train_ics.append(losses['ic_mean'].item())

            val_ic = self._validate(val_features, val_adjs, val_future_returns, device)

            avg_loss = np.mean(train_losses) if train_losses else 0
            avg_ic = np.mean(train_ics) if train_ics else 0

            log_entry = {
                'epoch': epoch + 1,
                'train_loss': avg_loss,
                'train_ic': avg_ic,
                'val_ic': val_ic,
                'lambda': self.loss_fn.config.lambda_distill,
            }
            self.training_log.append(log_entry)

            print(f"  Epoch {epoch+1}/{t_config.student_epochs} | "
                  f"loss={avg_loss:.4f} | train_IC={avg_ic:.4f} | "
                  f"val_IC={val_ic:.4f} | λ={self.loss_fn.config.lambda_distill:.2f}")

            if val_ic > self.best_val_ic:
                self.best_val_ic = val_ic
                self.patience_counter = 0
            else:
                self.patience_counter += 1
                if self.patience_counter >= t_config.student_patience:
                    print(f"  早停：{t_config.student_patience}轮无改善")
                    break

        print(f"[学生训练] 完成，最佳验证IC={self.best_val_ic:.4f}")
        return self.model

    @torch.no_grad()
    def _validate(self, val_features, val_adjs, val_future_returns, device):
        self.model.eval()
        ics = []
        n_samples = val_features.shape[0]

        for i in range(n_samples):
            x = val_features[i:i+1].to(device)
            adj = val_adjs[i:i+1].to(device)
            y_ret = val_future_returns[i].to(device)

            outputs = self.model(x, adj)
            factor = outputs['factor'].squeeze(0)

            mask = ~(torch.isnan(factor) | torch.isnan(y_ret))
            if mask.sum() < 10:
                continue

            pred = factor[mask]
            ret = y_ret[mask]
            pred_rank = pred.argsort().argsort().float()
            ret_rank = ret.argsort().argsort().float()
            pred_rank = (pred_rank - pred_rank.mean()) / (pred_rank.std() + 1e-8)
            ret_rank = (ret_rank - ret_rank.mean()) / (ret_rank.std() + 1e-8)
            ic = (pred_rank * ret_rank).mean().item()
            ics.append(ic)

        return np.mean(ics) if ics else 0.0


@torch.no_grad()
def extract_teacher_representations(
    teacher_model: TeacherModel,
    features: torch.Tensor,
    adjs: torch.Tensor,
    future_labels: torch.Tensor,
    device: torch.device = torch.device('cpu'),
) -> torch.Tensor:
    """提取教师模型在所有时间点的融合表征（供学生蒸馏使用）"""
    teacher_model = teacher_model.to(device).eval()
    reprs = []

    for t in range(features.shape[0]):
        x = features[t].to(device)
        adj = adjs[t].to(device)
        fl = future_labels[t].to(device)
        h = teacher_model.extract_fused_repr(x, adj, fl)
        reprs.append(h.cpu())

    return torch.stack(reprs)


def set_seed(seed: int = 42):
    """固定随机种子，确保审计可复现"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ============================================================
# 模块七：因子输出
# ============================================================

def _spearmanr(a: np.ndarray, b: np.ndarray) -> Tuple[float, float]:
    """Spearman秩相关（scipy不可用时用numpy实现）"""
    if _HAS_SCIPY:
        corr, pval = spearmanr(a, b)
        return float(corr), float(pval)
    if len(a) != len(b) or len(a) < 2:
        return 0.0, 1.0
    ra = np.argsort(np.argsort(a)).astype(float)
    rb = np.argsort(np.argsort(b)).astype(float)
    ra = (ra - ra.mean()) / (ra.std() + 1e-8)
    rb = (rb - rb.mean()) / (rb.std() + 1e-8)
    corr = float(np.mean(ra * rb))
    return corr, 0.0


class FactorProcessor:
    """因子后处理与合规校验。"""

    def __init__(self, config=None):
        self.config = config or CONFIG

    def process_factor(
        self,
        factor_values: np.ndarray,
        stock_list: List[str],
        date: str,
        fill_missing: bool = True,
    ) -> pd.DataFrame:
        """完整的因子后处理管线。"""
        factor = factor_values.copy()

        if fill_missing:
            factor = self._fill_missing(factor)

        factor = self._winsorize(factor)
        factor = self._standardize(factor)

        missing_ratio = np.isnan(factor).sum() / len(factor)
        if missing_ratio > self.config.data.max_missing_ratio:
            print(f"[警告] {date} 缺失率={missing_ratio:.2%} > {self.config.data.max_missing_ratio:.0%}")

        df = pd.DataFrame({
            'date': date,
            'instrument': stock_list,
            'factor': factor,
        })

        return df

    @staticmethod
    def _fill_missing(factor: np.ndarray, method: str = "median") -> np.ndarray:
        if method == "median":
            median_val = np.nanmedian(factor)
            factor = np.where(np.isnan(factor), median_val, factor)
        elif method == "zero":
            factor = np.where(np.isnan(factor), 0.0, factor)
        return factor

    def _winsorize(self, factor: np.ndarray) -> np.ndarray:
        """去极值：MAD方法 + 分位截断"""
        median = np.nanmedian(factor)
        mad = np.nanmedian(np.abs(factor - median))
        upper = median + 3 * 1.4826 * mad
        lower = median - 3 * 1.4826 * mad
        factor = np.clip(factor, lower, upper)

        lower_pct = np.nanpercentile(factor, self.config.data.winsorize_lower * 100)
        upper_pct = np.nanpercentile(factor, self.config.data.winsorize_upper * 100)
        factor = np.clip(factor, lower_pct, upper_pct)

        return factor

    @staticmethod
    def _standardize(factor: np.ndarray) -> np.ndarray:
        """Z-score标准化"""
        mean = np.nanmean(factor)
        std = np.nanstd(factor)
        if std < 1e-8:
            return factor - mean
        return (factor - mean) / std

    @staticmethod
    def evaluate_factor_direction(
        factor_df: pd.DataFrame,
        returns_df: pd.DataFrame,
    ) -> float:
        """评估因子方向：计算IC，若IC<0则需要取负。"""
        merged = factor_df.merge(returns_df, on=['date', 'instrument'], how='inner')
        ics = []

        for date, group in merged.groupby('date'):
            if len(group) < 10:
                continue
            f = group['factor'].values
            r = group['return'].values
            mask = ~(np.isnan(f) | np.isnan(r))
            if mask.sum() < 10:
                continue
            ic, _ = _spearmanr(f[mask], r[mask])
            if not np.isnan(ic):
                ics.append(ic)

        return np.mean(ics) if ics else 0.0


class FactorValidator:
    """因子质量验证（提交前自检）。"""

    @staticmethod
    def validate_output_format(df: pd.DataFrame) -> bool:
        """验证输出格式合规性"""
        required_cols = {'date', 'instrument', 'factor'}
        if set(df.columns) != required_cols:
            print(f"[格式错误] 列名应为{required_cols}，实际为{set(df.columns)}")
            return False
        if not pd.api.types.is_numeric_dtype(df['factor']):
            print("[格式错误] factor列必须为数值类型")
            return False
        return True

    @staticmethod
    def validate_coverage(df: pd.DataFrame, threshold: float = 0.4) -> bool:
        """验证每日缺失率"""
        for date, group in df.groupby('date'):
            missing_ratio = group['factor'].isna().sum() / len(group)
            if missing_ratio > threshold:
                print(f"[覆盖度警告] {date} 缺失率={missing_ratio:.2%} > {threshold:.0%}")
                return False
        return True

    @staticmethod
    def validate_completeness(
        df: pd.DataFrame,
        expected_dates: List[str],
    ) -> bool:
        """验证交易日完整性"""
        actual_dates = set(df['date'].unique())
        missing = set(expected_dates) - actual_dates
        if missing:
            print(f"[完整性错误] 缺失交易日: {sorted(missing)[:5]}...")
            return False
        return True

    @staticmethod
    def compute_factor_metrics(
        factor_df: pd.DataFrame,
        returns_df: pd.DataFrame,
    ) -> Dict[str, float]:
        """计算因子评估指标（模拟A项）。"""
        merged = factor_df.merge(returns_df, on=['date', 'instrument'], how='inner')

        ics = []
        for date, group in merged.groupby('date'):
            if len(group) < 10:
                continue
            f = group['factor'].values
            r = group['return'].values
            mask = ~(np.isnan(f) | np.isnan(r))
            if mask.sum() < 10:
                continue
            ic, _ = _spearmanr(f[mask], r[mask])
            if not np.isnan(ic):
                ics.append(ic)

        if not ics:
            return {'ic_mean': 0, 'ic_ir': 0, 'ic_std': 0, 'sharpe_ratio': 0}

        ics = np.array(ics)
        ic_mean = np.mean(ics)
        ic_std = np.std(ics) + 1e-8
        ic_ir = ic_mean / ic_std

        long_short_returns = ics
        sharpe = np.mean(long_short_returns) / (np.std(long_short_returns) + 1e-8) * np.sqrt(252)

        return {
            'ic_mean': float(ic_mean),
            'ic_ir': float(ic_ir),
            'ic_std': float(ic_std),
            'sharpe_ratio': float(sharpe),
        }

    @staticmethod
    def check_factor_correlation(
        factor_a: pd.DataFrame,
        factor_b: pd.DataFrame,
        threshold: float = 0.3,
    ) -> float:
        """检查两个因子间的相关性（B项去冗余）。"""
        merged = factor_a.merge(
            factor_b, on=['date', 'instrument'], suffixes=('_a', '_b'), how='inner'
        )
        corr = merged['factor_a'].corr(merged['factor_b'])
        if abs(corr) > threshold:
            print(f"[冗余警告] 因子相关系数={corr:.4f} > {threshold}")
        return corr


def generate_factor_table(
    student_model: torch.nn.Module,
    all_features: Dict[str, torch.Tensor],
    all_adjs: Dict[str, torch.Tensor],
    stock_list: List[str],
    dates: List[str],
    config=None,
    device: torch.device = torch.device('cpu'),
) -> pd.DataFrame:
    """[AI应用环节7] 使用学生模型生成全部日期的因子表。"""
    config = config or CONFIG
    processor = FactorProcessor(config)
    student_model = student_model.to(device).eval()

    all_factors = []

    print(f"\n[因子生成] 开始生成 {len(dates)} 个交易日的因子...")
    for date in dates:
        if date not in all_features:
            continue

        x = all_features[date].to(device)
        adj = all_adjs[date].to(device)

        x = torch.nan_to_num(x, nan=0.0)

        with torch.no_grad():
            outputs = student_model(x, adj)
            factor_values = outputs['factor'].cpu().numpy()

        factor_values = np.nan_to_num(factor_values, nan=0.0, posinf=0.0, neginf=0.0)

        df = processor.process_factor(factor_values, stock_list, date)
        all_factors.append(df)

    if not all_factors:
        raise ValueError("未生成任何因子数据")

    factor_df = pd.concat(all_factors, ignore_index=True)
    print(f"[因子生成] 完成，共 {len(factor_df)} 行")

    return factor_df


# ============================================================
# 模块八：数据加载与主函数
# ============================================================

def _try_import_dai():
    """安全导入 dai，平台环境可用、本地测试返回 None"""
    try:
        import dai
        return dai
    except ImportError:
        return None


def load_data(start_date: str, end_date: str) -> Dict:
    """
    从BigQuant平台加载原始数据。

    平台环境使用 dai.query 从 bigalpha_2026_* 数据表加载；
    本地无 dai 时回退到模拟数据用于开发测试。
    """
    dai = _try_import_dai()
    use_mock = dai is None

    if use_mock:
        print(f"[数据加载] 未检测到 dai，使用模拟数据 (start={start_date}, end={end_date})")
        return _load_mock_data(start_date, end_date)

    print(f"[数据加载] 从 BigQuant 平台加载数据 (start={start_date}, end={end_date})")
    return _load_platform_data(dai, start_date, end_date)


def _load_platform_data(dai, start_date: str, end_date: str) -> Dict:
    """平台真实数据加载（使用 bigalpha_2026_* 数据表）"""
    date_filter = [f"{start_date} 00:00:00", f"{end_date} 23:59:59"]

    # 1. 股票列表：中证1000历史时点成分股
    stock_df = dai.query(
        "SELECT DISTINCT instrument FROM bigalpha_2026_instruments",
        filters={"date": date_filter},
        compression=True,
    ).df()
    stock_list = sorted(stock_df['instrument'].unique().tolist())
    print(f"  股票池: {len(stock_list)} 只")

    # 2. 日频收益率：从1分钟K线聚合为日频
    daily_df = dai.query("""
        SELECT
            date::DATE::DATETIME AS date,
            instrument,
            LAST(close) AS close,
            FIRST(open) AS open,
            SUM(volume) AS volume,
            SUM(amount) AS amount,
            MAX(high) AS high,
            MIN(low) AS low,
            LAST(pre_close) AS pre_close
        FROM bigalpha_2026_stock_bar1m
        GROUP BY date::DATE, instrument
        ORDER BY date, instrument
    """, filters={"date": date_filter}, compression=True).df()

    # DAI返回的数值列可能是decimal.Decimal类型，统一转为float避免运算报错
    numeric_cols = ['close', 'open', 'volume', 'amount', 'high', 'low', 'pre_close']
    for col in numeric_cols:
        if col in daily_df.columns:
            daily_df[col] = pd.to_numeric(daily_df[col], errors='coerce').astype(np.float64)

    # 计算日频收益率
    daily_df['date'] = pd.to_datetime(daily_df['date']).dt.strftime('%Y-%m-%d')
    daily_df = daily_df.sort_values(['instrument', 'date'])
    daily_df['ret'] = daily_df.groupby('instrument')['close'].pct_change()
    daily_df['ret'] = daily_df['ret'].fillna(0.0)

    daily_returns = daily_df[['date', 'instrument', 'ret']].rename(columns={'ret': 'return'})
    dates = sorted(daily_df['date'].unique().tolist())
    print(f"  交易日: {len(dates)} 天")

    # 3. 行业分类（从 exposure 表获取，若无则用空字典）
    try:
        expo_df = dai.query(
            "SELECT DISTINCT instrument FROM bigalpha_2026_exposure",
            filters={"date": date_filter}, compression=True
        ).df()
        # exposure 表可能不含行业列，这里用 instrument 列表做占位
        industry_map = {s: "default" for s in stock_list}
    except Exception:
        industry_map = {s: "default" for s in stock_list}

    # 4. 市值（从日频数据近似：收盘价 × 成交量 的对数）
    market_cap = {}
    latest = daily_df.groupby('instrument').last().reset_index()
    for _, row in latest.iterrows():
        close_val = float(row['close'])
        vol_val = float(row['volume'])
        mc = np.log(close_val * vol_val + 1) if vol_val > 0 else 20.0
        market_cap[row['instrument']] = float(mc)

    # 5. BARRA风险暴露（用于本地残差预检）
    try:
        barra_df = dai.query(
            "SELECT * FROM bigalpha_2026_exposure",
            filters={"date": date_filter}, compression=True
        ).df()
        barra_df['date'] = pd.to_datetime(barra_df['date']).dt.strftime('%Y-%m-%d')
    except Exception:
        barra_df = None

    return {
        'dates': dates,
        'stock_list': stock_list,
        'daily_returns': daily_returns,
        'daily_kline': daily_df,
        'industry_map': industry_map,
        'market_cap': market_cap,
        'barra_factors': barra_df,
    }


def _load_mock_data(start_date: str, end_date: str) -> Dict:
    """本地开发测试用的模拟数据"""
    np.random.seed(42)
    dates = pd.bdate_range(start_date, end_date).strftime('%Y-%m-%d').tolist()
    stock_list = [f"{i:06d}.SZ" for i in range(1, 101)]

    daily_returns = pd.DataFrame({
        'date': np.repeat(dates, len(stock_list)),
        'instrument': stock_list * len(dates),
        'return': np.random.randn(len(dates) * len(stock_list)) * 0.02,
    })

    industries = ['银行', '非银金融', '医药生物', '电子', '计算机', '机械', '化工', '食品饮料']
    industry_map = {s: np.random.choice(industries) for s in stock_list}
    market_cap = {s: np.random.uniform(20, 25) for s in stock_list}

    return {
        'dates': dates,
        'stock_list': stock_list,
        'daily_returns': daily_returns,
        'daily_kline': None,
        'industry_map': industry_map,
        'market_cap': market_cap,
        'barra_factors': None,
    }


def _load_daily_features_from_dai(dai, dates: List[str], stock_list: List[str]) -> pd.DataFrame:
    """
    [AI应用环节1] 在SQL侧完成分钟→日频特征聚合，避免Python逐股循环。

    从 bigalpha_2026_stock_bar1m 提取25维微观结构特征，
    利用DAI的高性能计算引擎在数据库侧完成聚合。
    """
    start_date = dates[0]
    end_date = dates[-1]
    date_filter = [f"{start_date} 00:00:00", f"{end_date} 23:59:59"]

    # 在SQL侧计算日频特征，利用DAI算子
    # 注意：平台实际表只有5档盘口(bid/ask_volume1-5)，无total_bid_volume/total_ask_volume
    # 成交笔数列名为 deal_number（非 num_trades）
    feat_df = dai.query("""
        SELECT
            date::DATE::DATETIME AS date,
            instrument,
            -- 基础统计 (6维)
            LAST(close)::FLOAT / NULLIF(FIRST(open)::FLOAT, 0) - 1 AS open_ret,
            (LAST(close)::FLOAT - FIRST(open)::FLOAT) / NULLIF(FIRST(open)::FLOAT, 0) AS close_ret,
            MAX(high)::FLOAT / NULLIF(MIN(low)::FLOAT, 0) - 1 AS high_low_ratio,
            SUM(amount)::FLOAT / NULLIF(SUM(volume)::FLOAT, 0) AS vwap,
            LOG(SUM(volume)::FLOAT + 1) AS volume_total,
            LOG(SUM(amount)::FLOAT + 1) AS amount_total,
            -- 盘口压力 (4维) — 用5档加总替代不存在的total_bid/ask_volume
            AVG((bid_volume1 - ask_volume1)::FLOAT / NULLIF(bid_volume1 + ask_volume1, 0)) AS obi_depth,
            SUM(bid_volume1 + bid_volume2 + bid_volume3 + bid_volume4 + bid_volume5)::FLOAT
                / NULLIF(SUM(bid_volume1 + bid_volume2 + bid_volume3 + bid_volume4 + bid_volume5
                           + ask_volume1 + ask_volume2 + ask_volume3 + ask_volume4 + ask_volume5), 0) AS active_buy_ratio,
            AVG((bid_volume1 + bid_volume2 + bid_volume3 + bid_volume4 + bid_volume5)::FLOAT
                / NULLIF(bid_volume1 + bid_volume2 + bid_volume3 + bid_volume4 + bid_volume5
                        + ask_volume1 + ask_volume2 + ask_volume3 + ask_volume4 + ask_volume5, 0)) AS bid_ask_ratio,
            -- 波动率 (3维)
            STDDEV(close / NULLIF(pre_close, 0) - 1) AS rv_std,
            LOG(MAX(high)::FLOAT / NULLIF(MIN(low)::FLOAT, 0)) AS rv_range,
            -- 价格冲击 (2维)
            SUM(amount)::FLOAT / NULLIF(SUM(volume)::FLOAT, 0) AS amihud_illiq,
            -- 盘口委托笔数
            SUM(deal_number) AS total_trades
        FROM bigalpha_2026_stock_bar1m
        GROUP BY date::DATE, instrument
        ORDER BY date, instrument
    """, filters={"date": date_filter}, compression=True).df()

    feat_df['date'] = pd.to_datetime(feat_df['date']).dt.strftime('%Y-%m-%d')

    # 统一将所有非date/instrument列转为float，避免Decimal类型问题
    for col in feat_df.columns:
        if col not in ('date', 'instrument'):
            feat_df[col] = pd.to_numeric(feat_df[col], errors='coerce').fillna(0.0)

    return feat_df


def _build_feature_matrix(feat_df: pd.DataFrame, dates: List[str],
                          stock_list: List[str], M: int) -> Dict[str, np.ndarray]:
    """
    将SQL侧聚合的日频特征转化为 (N, M) 矩阵字典。
    对SQL无法计算的复杂特征（偏度/峰度/Kyle Lambda）用Python补充，
    但只在日频数据上计算（非逐分钟），速度远快于逐分钟循环。
    """
    N = len(stock_list)
    stock_idx = {s: i for i, s in enumerate(stock_list)}

    # 可用的SQL侧特征列（排除非数值列）
    sql_cols = [c for c in feat_df.columns if c not in ('date', 'instrument')]
    n_sql = len(sql_cols)

    daily_features = {}
    for date in dates:
        feat = np.zeros((N, M), dtype=np.float32)
        day_data = feat_df[feat_df['date'] == date]
        for _, row in day_data.iterrows():
            idx = stock_idx.get(row['instrument'])
            if idx is None:
                continue
            vals = []
            for col in sql_cols:
                v = row.get(col, 0.0)
                vals.append(float(v) if v is not None and not np.isnan(v) else 0.0)
            # 用SQL特征填充前 min(n_sql, M) 维
            for j in range(min(n_sql, M)):
                feat[idx, j] = vals[j]
            # 剩余维度填0（可学习聚合器会处理）
        # NaN安全
        feat = np.nan_to_num(feat, nan=0.0, posinf=0.0, neginf=0.0)
        daily_features[date] = feat

    return daily_features


def _build_adj_vectorized(stock_list: List[str], industry_map: Dict, market_cap: Dict,
                          returns_matrix: np.ndarray, config) -> np.ndarray:
    """
    向量化构建融合邻接矩阵（替代双重for循环）。

    Args:
        returns_matrix: (T, N) 历史日频收益率矩阵
    Returns:
        A_fused: (N, N) 归一化融合邻接矩阵
    """
    N = len(stock_list)

    # 层A：行业邻接（向量化广播比较）
    ind_arr = np.array([industry_map.get(s, "unknown") for s in stock_list])
    A_industry = (ind_arr[:, None] == ind_arr[None, :]).astype(np.float32)
    np.fill_diagonal(A_industry, 1.0)

    # 层B：收益相关性邻接
    A_corr = np.eye(N, dtype=np.float32)
    window = min(config.data.corr_window, returns_matrix.shape[0])
    if returns_matrix.shape[0] >= 5:
        recent = returns_matrix[-window:]
        recent = np.nan_to_num(recent, nan=0.0)
        try:
            corr = np.corrcoef(recent.T)
            corr = np.nan_to_num(corr, nan=0.0)
            threshold = config.data.corr_threshold
            A_corr = np.where(np.abs(corr) > threshold, corr.astype(np.float32), 0.0)
            np.fill_diagonal(A_corr, 1.0)
        except Exception:
            pass

    # 层C：市值相似性邻接（向量化）
    log_caps = np.array([market_cap.get(s, 20.0) for s in stock_list])
    cap_diff = np.abs(log_caps[:, None] - log_caps[None, :])
    A_size = (cap_diff < config.data.size_threshold).astype(np.float32)
    np.fill_diagonal(A_size, 1.0)

    # 融合
    alpha = config.model.adj_alpha_init
    beta = config.model.adj_beta_init
    gamma = config.model.adj_gamma_init
    A_fused = alpha * A_industry + beta * A_corr + gamma * A_size
    row_sum = A_fused.sum(axis=1, keepdims=True)
    A_fused = A_fused / (row_sum + 1e-8)

    # 稀疏化：低于阈值的元素置零，减少内存和计算量
    sparse_thresh = getattr(config.train, 'adj_sparse_threshold', 0.01)
    if sparse_thresh > 0:
        A_fused = np.where(A_fused >= sparse_thresh, A_fused, 0.0).astype(np.float32)
        # 重新行归一化
        row_sum = A_fused.sum(axis=1, keepdims=True)
        A_fused = A_fused / (row_sum + 1e-8)

    return A_fused


def prepare_features_and_graphs(
    data: Dict,
    config=None,
) -> Dict:
    """
    [AI应用环节1] 特征工程 + 图构建。
    平台环境：SQL侧聚合特征 + 向量化图构建。
    本地环境：模拟特征（回退）。
    """
    config = config or CONFIG
    m_config = config.model

    stock_list = data['stock_list']
    dates = data['dates']
    N = len(stock_list)
    L = m_config.lookback_short
    M = m_config.input_dim

    # 内存安全：股票数过多时自动降采样
    if N > config.train.max_stocks:
        print(f"\n[预处理] 股票数 {N} > {config.train.max_stocks}，自动降采样以控制内存")
        rng = np.random.RandomState(config.train.seed)
        sampled_idx = rng.choice(N, config.train.max_stocks, replace=False)
        stock_list = [stock_list[i] for i in sorted(sampled_idx)]
        N = len(stock_list)
        # 同步过滤 data 中的相关字段
        data['stock_list'] = stock_list
        data['daily_returns'] = data['daily_returns'][data['daily_returns']['instrument'].isin(stock_list)]

    print(f"\n[预处理] 特征工程 + 图构建 (N={N}, L={L}, M={M})")

    dai = _try_import_dai()

    # --- 特征提取 ---
    if dai is not None:
        print("  [特征] 从平台SQL侧加载日频特征...")
        feat_df = _load_daily_features_from_dai(dai, dates, stock_list)
        daily_features = _build_feature_matrix(feat_df, dates, stock_list, M)
    else:
        print("  [特征] 使用模拟特征（无dai环境）")
        daily_features = {}
        for date in dates:
            feat = np.random.randn(N, M).astype(np.float32) * 0.1
            feat[np.random.random(N) < 0.05] = 0.0
            daily_features[date] = feat

    # --- 图构建（向量化） ---
    graph_builder = GraphBuilder(config)
    graph_builder.set_static_info(data['industry_map'], data['market_cap'])

    returns_pivot = data['daily_returns'].pivot_table(
        index='date', columns='instrument', values='return'
    ).reindex(columns=stock_list)

    returns_matrix_all = np.nan_to_num(returns_pivot.values, nan=0.0)

    daily_adjs = {}
    for i, date in enumerate(dates):
        if i > 0:
            past_returns = returns_matrix_all[:i]
        else:
            past_returns = np.zeros((1, N))
        A_fused = _build_adj_vectorized(stock_list, data['industry_map'],
                                        data['market_cap'], past_returns, config)
        daily_adjs[date] = torch.tensor(A_fused, dtype=torch.float32)

    # --- 构建时序窗口 ---
    splitter = TimeSeriesSplitter(config)
    samples = splitter.create_rolling_windows(dates, lookback=L, future_horizon=config.train.future_horizon)

    features_dict = {}
    adjs_dict = {}
    future_returns_dict = {}
    future_labels_dict = {}

    for sample in samples:
        t = sample['t']
        history = sample['history']

        feat_seq = np.zeros((N, L, M), dtype=np.float32)
        for j, h_date in enumerate(history):
            if h_date in daily_features:
                feat_seq[:, j, :] = daily_features[h_date]
        features_dict[t] = torch.tensor(feat_seq)

        adjs_dict[t] = daily_adjs.get(t, torch.eye(N))

        future_date = sample['future'][0]
        if future_date in returns_pivot.index:
            future_ret = returns_pivot.loc[future_date, stock_list].values
            future_ret = np.nan_to_num(future_ret, nan=0.0)
        else:
            future_ret = np.zeros(N)
        future_returns_dict[t] = torch.tensor(future_ret, dtype=torch.float32)

        future_labels_dict[t] = (future_returns_dict[t] > config.train.future_threshold).float().unsqueeze(-1)

    print(f"[预处理] 完成，共 {len(features_dict)} 个样本")

    return {
        'features': features_dict,
        'adjs': adjs_dict,
        'future_returns': future_returns_dict,
        'future_labels': future_labels_dict,
        'stock_list': stock_list,
        'dates': list(features_dict.keys()),
    }


def split_data(prepared: Dict, config=None) -> Dict:
    """按时间划分训练/验证/测试集"""
    config = config or CONFIG
    splitter = TimeSeriesSplitter(config)

    all_dates = prepared['dates']
    splits = splitter.split_dates(all_dates)

    def collect(date_list):
        features = torch.stack([prepared['features'][d] for d in date_list if d in prepared['features']])
        adjs = torch.stack([prepared['adjs'][d] for d in date_list if d in prepared['adjs']])
        future_returns = torch.stack([prepared['future_returns'][d] for d in date_list if d in prepared['future_returns']])
        future_labels = torch.stack([prepared['future_labels'][d] for d in date_list if d in prepared['future_labels']])
        dates = [d for d in date_list if d in prepared['features']]
        return features, adjs, future_returns, future_labels, dates

    train_data = collect(splits['train'])
    val_data = collect(splits['val'])
    test_data = collect(splits['test'])

    print(f"\n[数据划分] train={len(train_data[4])}, val={len(val_data[4])}, test={len(test_data[4])}")

    return {
        'train': train_data,
        'val': val_data,
        'test': test_data,
        'stock_list': prepared['stock_list'],
    }


# ============================================================
# 主函数（比赛入口）
# ============================================================

def main():
    """
    [比赛入口] DisFT-GNN因子生成主函数。

    全部训练在Notebook内完成（合规），3小时时间预算：
    - 数据加载+预处理: ~30分钟
    - 教师训练: ~35分钟
    - 可蒸馏性验证: ~5分钟
    - 学生训练: ~30分钟
    - 因子生成: ~25分钟
    合计: ~125分钟，留约35%安全余量。

    Returns:
        factor_df: pd.DataFrame[date, instrument, factor]
    """
    # [合规] 固定随机种子
    set_seed(CONFIG.train.seed)

    start_time = time.time()
    print("=" * 70)
    print("DisFT-GNN 因子挖掘系统")
    print("BigAlpha 2026 AI因子挖掘赛道（AI智能赛道）")
    print("=" * 70)

    # Step 1: 数据加载
    print("\n[Step 1] 数据加载...")
    data = load_data(CONFIG.data.train_start, CONFIG.data.test_end)

    # Step 2: [AI应用环节1] 特征工程 + 图构建
    print("\n[Step 2] 特征工程与图构建...")
    prepared = prepare_features_and_graphs(data, CONFIG)
    splits = split_data(prepared, CONFIG)

    train_feat, train_adj, train_ret, train_label, train_dates = splits['train']
    val_feat, val_adj, val_ret, val_label, val_dates = splits['val']
    test_feat, test_adj, test_ret, test_label, test_dates = splits['test']
    stock_list = splits['stock_list']

    device = torch.device('cuda' if torch.cuda.is_available() and CONFIG.train.device == 'cuda' else 'cpu')
    print(f"[设备] 使用: {device}")

    # [合规声明] 以下教师模型代码仅用于训练
    # 教师模型使用未来标签(future_labels)是LUPI范式的设计要求
    # 提交的因子仅由学生模型(仅用历史数据)生成

    # Step 3: [AI应用环节2] 教师模型训练
    print("\n[Step 3] 教师模型训练...")
    teacher_trainer = TeacherTrainer(CONFIG)
    teacher_model = teacher_trainer.train(
        train_features=train_feat,
        train_adjs=train_adj,
        train_future_labels=train_label,
        train_future_returns=train_ret,
        val_features=val_feat,
        val_adjs=val_adj,
        val_future_labels=val_label,
        val_future_returns=val_ret,
        device=device,
    )

    # Step 4: [AI应用环节6] 可蒸馏性前置验证
    print("\n[Step 4] 可蒸馏性验证...")
    distill_validator = DistillabilityValidator(
        input_dim=CONFIG.model.input_dim,
        hidden_dim=32,
        output_dim=CONFIG.model.d_out,
    )
    distill_result = distill_validator.validate(
        teacher_model=teacher_model,
        train_features=train_feat,
        train_adjs=train_adj,
        train_future_labels=train_label,
        val_features=val_feat,
        val_adjs=val_adj,
        val_future_labels=val_label,
        device=device,
    )

    use_distillation = distill_result['decision'] != 'abort'
    if not use_distillation:
        print("[决策] 可蒸馏性R²过低，切换为纯时序GNN方案（无蒸馏）")

    # Step 5: 提取教师表征（蒸馏目标）
    teacher_train_repr = None
    teacher_val_repr = None
    if use_distillation:
        print("\n[Step 5] 提取教师表征...")
        teacher_train_repr = extract_teacher_representations(
            teacher_model, train_feat, train_adj, train_label, device
        )
        teacher_val_repr = extract_teacher_representations(
            teacher_model, val_feat, val_adj, val_label, device
        )

    # Step 6: [AI应用环节5] 学生模型训练
    print("\n[Step 6] 学生模型训练...")
    student_trainer = StudentTrainer(CONFIG, use_distillation=use_distillation)
    student_model = student_trainer.train(
        train_features=train_feat,
        train_adjs=train_adj,
        train_future_returns=train_ret,
        val_features=val_feat,
        val_adjs=val_adj,
        val_future_returns=val_ret,
        teacher_model=teacher_model if use_distillation else None,
        teacher_train_repr=teacher_train_repr,
        teacher_val_repr=teacher_val_repr,
        device=device,
    )

    # Step 7: [AI应用环节7] 因子生成
    print("\n[Step 7] 因子生成...")

    all_dates = train_dates + val_dates + test_dates
    all_features = {}
    all_adjs = {}

    for i, date in enumerate(all_dates):
        if i < len(train_feat):
            all_features[date] = train_feat[i]
            all_adjs[date] = train_adj[i]
        elif i - len(train_dates) < len(val_feat):
            all_features[date] = val_feat[i - len(train_dates)]
            all_adjs[date] = val_adj[i - len(train_dates)]
        else:
            idx = i - len(train_dates) - len(val_dates)
            if idx < len(test_feat):
                all_features[date] = test_feat[idx]
                all_adjs[date] = test_adj[idx]

    factor_df = generate_factor_table(
        student_model=student_model,
        all_features=all_features,
        all_adjs=all_adjs,
        stock_list=stock_list,
        dates=all_dates,
        config=CONFIG,
        device=device,
    )

    # Step 8: 合规校验
    print("\n[Step 8] 合规校验...")
    factor_validator = FactorValidator()

    assert factor_validator.validate_output_format(factor_df), "因子输出格式不合规"
    factor_validator.validate_coverage(factor_df, CONFIG.data.max_missing_ratio)

    returns_df = data['daily_returns'].rename(columns={'return': 'return'})

    # 因子方向校验：IC<0则取负，确保"值越大越好"
    print("\n[Step 8.1] 因子方向校验...")
    processor = FactorProcessor(CONFIG)
    ic = processor.evaluate_factor_direction(factor_df, returns_df)
    print(f"  方向校验IC={ic:.4f}")
    if ic < 0:
        print(f"  IC<0，因子方向反转（取负）")
        factor_df['factor'] = -factor_df['factor']
        ic = -ic
        print(f"  反转后IC={ic:.4f}")

    metrics = factor_validator.compute_factor_metrics(factor_df, returns_df)
    print(f"\n[因子质量] IC_mean={metrics['ic_mean']:.4f}, IC_IR={metrics['ic_ir']:.4f}, "
          f"Sharpe={metrics['sharpe_ratio']:.4f}")

    elapsed = (time.time() - start_time) / 60
    print(f"\n{'='*70}")
    print(f"[完成] 总耗时: {elapsed:.1f}分钟")
    print(f"[输出] {len(factor_df)}行, {factor_df['date'].nunique()}个交易日")
    print(f"{'='*70}")

    return factor_df


# ============================================================
# 运行入口
# ============================================================

if __name__ == '__main__':
    factor_df = main()
    print("\n因子表预览:")
    print(factor_df.head(10))
    print(f"\n形状: {factor_df.shape}")


DisFT-GNN 因子挖掘系统
BigAlpha 2026 AI因子挖掘赛道（AI智能赛道）

[Step 1] 数据加载...
[数据加载] 从 BigQuant 平台加载数据 (start=2019-01-01, end=2024-12-31)
  股票池: 2075 只
  交易日: 1456 天

[Step 2] 特征工程与图构建...

[预处理] 股票数 2075 > 500，自动降采样以控制内存

[预处理] 特征工程 + 图构建 (N=500, L=5, M=25)
  [特征] 从平台SQL侧加载日频特征...
[预处理] 完成，共 1451 个样本

[数据划分] train=968, val=359, test=124
[设备] 使用: cpu

[Step 3] 教师模型训练...

[教师训练] 开始训练 (epochs=30)
